# 01 — Quickstart

Welcome to **checkllm**, the pytest of LLM testing.

This notebook walks through:

1. Installing checkllm.
2. Running your first deterministic check.
3. Running an LLM-judged metric **without a real API key**.
4. Interpreting the structured results.

Everything below is fully runnable offline — we plug in a recorded `FakeJudge` for the judged metrics. A cell near the bottom shows how to switch to a real provider.

## 1. Install

Uncomment if checkllm is not yet installed:

In [ ]:
# !pip install checkllm

## 2. Deterministic checks

Deterministic checks never call an LLM. They are the safest place to start — free, instant, and fully reproducible.

In [ ]:
from checkllm.deterministic import DeterministicChecks

det = DeterministicChecks()
output = "Python is a high-level programming language created by Guido van Rossum."

print("contains Python:", det.contains(output, "Python").passed)
print("contains Guido :", det.contains(output, "Guido").passed)
print("no JavaScript :", det.not_contains(output, "JavaScript").passed)
print("under 50 tokens:", det.max_tokens(output, limit=50).passed)

## 3. LLM-judged metrics with a FakeJudge

Judged metrics call an LLM to grade the response. To keep the notebook runnable offline we wire in a `FakeJudge` that returns a pre-recorded score.

In [ ]:
from checkllm.judge import JudgeResponse


class FakeJudge:
    """Deterministic in-process judge used to keep this notebook offline.

    Real runs should swap this for ``OpenAIJudge`` / ``AnthropicJudge`` etc.
    """

    def __init__(self, score: float = 0.9, reasoning: str = "looks good") -> None:
        self._score = score
        self._reasoning = reasoning

    async def evaluate(self, prompt: str, system_prompt: str | None = None) -> JudgeResponse:
        return JudgeResponse(
            score=self._score,
            reasoning=self._reasoning,
            raw_output=str(self._score),
            cost=0.0,
        )


judge = FakeJudge(score=0.9, reasoning="offline stub")

In [ ]:
# Jupyter supports top-level ``await``; outside notebooks use
# ``asyncio.run`` instead.
result = await judge.evaluate(prompt="Is the sky blue?")
print("score     :", result.score)
print("reasoning :", result.reasoning)
print("cost      :", result.cost)

## 4. Interpreting results

`JudgeResponse` is a Pydantic model. The fields you typically care about are:

| Field | Meaning |
|-------|---------|
| `score` | value in `[0.0, 1.0]`, higher is better |
| `reasoning` | textual justification from the judge |
| `cost` | estimated USD (0.0 for the offline stub) |

Pair the score with a threshold (commonly 0.7 or 0.8) and fail the surrounding test when the score falls below it.

## 5. Switch to a real provider

In [ ]:
# -------------------------------------------------------------------
# SWITCH TO A REAL PROVIDER
# -------------------------------------------------------------------
# Uncomment and replace the FakeJudge above once you have API keys:
#
# from checkllm.judge import OpenAIJudge
# judge = OpenAIJudge(api_key="sk-...", model="gpt-4o-mini")
#
# from checkllm.judge import AnthropicJudge
# judge = AnthropicJudge(api_key="...", model="claude-3-5-sonnet-20241022")
